# Olostep Tool Integration Examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/run-llama/llama_index/blob/main/llama-index-integrations/tools/llama-index-tools-olostep/examples/olostep.ipynb)

This notebook demonstrates how to use the Olostep tool integration with LlamaIndex agents for web scraping, crawling, searching, and AI-powered research.

## Installation

First, install the required packages:

In [ ]:
!pip install llama-index-tools-olostep openai python-dotenv

## Setup

Set up your API keys for Olostep and OpenAI:

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set up API keys
OLOSTEP_API_KEY = os.getenv("OLOSTEP_API_KEY", "your-olostep-api-key")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-openai-api-key")

# Verify keys are set
if OLOSTEP_API_KEY == "your-olostep-api-key":
    print("⚠️  Please set OLOSTEP_API_KEY environment variable")
if OPENAI_API_KEY == "your-openai-api-key":
    print("⚠️  Please set OPENAI_API_KEY environment variable")

## Example 1: Scrape a URL and Summarize

This example shows how to use an agent to scrape a webpage and summarize its content:

In [ ]:
from llama_index.tools.olostep import OlostepToolSpec
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

# Initialize the Olostep tool spec
tool_spec = OlostepToolSpec(api_key=OLOSTEP_API_KEY)

# Create an agent with the Olostep tools
agent = FunctionAgent(
    tools=tool_spec.to_tool_list(),
    llm=OpenAI(model="gpt-4o", api_key=OPENAI_API_KEY),
)

# Run the agent to scrape and summarize
result = await agent.run(
    "Scrape https://docs.olostep.com and provide a brief summary of the main documentation topics."
)

print("Agent Response:")
print(result)

## Example 2: Direct Scraping

Use the tool spec directly without an agent for simpler tasks:

In [ ]:
# Scrape a single URL
documents = tool_spec.scrape_url(
    url="https://example.com",
    formats="markdown",
    wait_before_scraping=2000,  # Wait 2 seconds for JS rendering
)

for doc in documents:
    print(f"URL: {doc.extra_info['url']}")
    print(f"Content (first 500 chars):")
    print(doc.text[:500])
    print("...")

## Example 3: Research Agent - Search and Answer Questions

Use the agent to conduct web research and synthesize answers:

In [ ]:
# Create an agent for research
research_agent = FunctionAgent(
    tools=tool_spec.to_tool_list(),
    llm=OpenAI(model="gpt-4o", api_key=OPENAI_API_KEY),
)

# Conduct research
research_result = await research_agent.run(
    "Research the current CEO of OpenAI and provide their background and key achievements."
)

print("Research Result:")
print(research_result)

## Example 4: Map Website Structure

Discover the structure of a website before scraping:

In [ ]:
# Map a website to discover its structure
map_docs = tool_spec.map_website(
    url="https://docs.olostep.com",
    top_n=20,  # Get top 20 URLs
)

print(f"Discovered {map_docs[0].extra_info['total_urls']} URLs")
print("\nFirst few URLs:")
urls = map_docs[0].text.split("\n")[:5]
for url in urls:
    print(f"  - {url}")

## Example 5: Crawl a Website

Crawl multiple pages from a website with filtering:

In [ ]:
# Crawl a website section
crawl_docs = tool_spec.crawl_website(
    url="https://docs.olostep.com",
    max_pages=10,
    include_urls="/guides/**,/api/**",  # Only crawl guide and API pages
)

print(f"Crawled {len(crawl_docs)} pages")
for i, doc in enumerate(crawl_docs[:3]):
    print(f"\nPage {i+1}: {doc.extra_info['url']}")
    print(f"Content preview: {doc.text[:200]}...")

## Example 6: Web Search

Search the web for relevant pages:

In [ ]:
# Search the web
search_results = tool_spec.search_web(query="best practices for web scraping")

print(f"Found {len(search_results)} results")
for i, doc in enumerate(search_results[:5]):
    print(f"\nResult {i+1}:")
    print(f"  URL: {doc.extra_info['url']}")
    print(f"  Content: {doc.text[:200]}...")

## Example 7: Answer Question

Get AI-synthesized answers with verified sources:

In [ ]:
# Get an AI-synthesized answer
answer_docs = tool_spec.answer_question(
    task="What are the latest developments in AI agents and their applications?"
)

print("Answer:")
print(answer_docs[0].text)
print("\nSources:")
for source in answer_docs[0].extra_info.get("sources", []):
    print(f"  - {source}")

## Example 8: Batch Scraping

Scrape multiple URLs efficiently in a batch:

In [ ]:
# Batch scrape multiple URLs
urls_to_scrape = [
    "https://example.com/page1",
    "https://example.com/page2",
    "https://example.com/page3",
]

batch_docs = tool_spec.batch_scrape(
    urls=",".join(urls_to_scrape),
    formats="markdown",
)

print(f"Batch scraped {len(batch_docs)} URLs")
for doc in batch_docs:
    print(f"  - {doc.extra_info['url']}: {len(doc.text)} characters")

## Example 9: Structured Data Extraction

Extract structured data using pre-built parsers:

In [ ]:
# Extract structured data using a parser
product_docs = tool_spec.scrape_url(
    url="https://www.amazon.com/dp/B0BSQJ7YNR",
    formats="json",
    parser_id="@olostep/amazon-it-product",
)

print("Extracted Product Data:")
print(product_docs[0].text)

## Example 10: Complete Research Pipeline

Combine multiple tools for a complete research workflow:

In [ ]:
# Complete pipeline: search -> map -> crawl -> synthesize

# Step 1: Search for relevant pages
search_results = tool_spec.search_web(query="Olostep web scraping API documentation")
print(f"Step 1: Found {len(search_results)} search results")

# Step 2: Extract the best result
if search_results:
    best_url = search_results[0].extra_info["url"]
    print(f"Step 2: Selected URL: {best_url}")

# Step 3: Map the website structure
map_results = tool_spec.map_website(url=best_url, top_n=10)
discovered_urls = [url for url in map_results[0].text.split("\n") if url]
print(f"Step 3: Discovered {len(discovered_urls)} URLs")

# Step 4: Crawl the main page and important sections
if len(discovered_urls) > 0:
    crawl_results = tool_spec.crawl_website(
        url=best_url,
        max_pages=5,
    )
    print(f"Step 4: Crawled {len(crawl_results)} pages")

# Step 5: Use agent to synthesize information
synthesis_agent = FunctionAgent(
    tools=tool_spec.to_tool_list(),
    llm=OpenAI(model="gpt-4o", api_key=OPENAI_API_KEY),
)

synthesis_result = await synthesis_agent.run(
    "Based on the information gathered, provide a comprehensive summary of what Olostep is and how to use its API."
)

print("\nStep 5: Synthesis Complete")
print("="*50)
print(synthesis_result)